In [ ]:
import pandas as pd
import os
import sys
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
best_pair = (2, 11)
final_design = pd.read_csv(f'Ahsa1-{best_pair}_pattern_design_siRNA_N=85.csv', index_col=0)

text = ""
for si in final_design.siRNA:
    text += f">{si}\n{si}\n"
with open('Ahsa_siRNA.fasta', 'w') as f:
    f.write(text)

In [ ]:
# 1) Point this at wherever your blastn binary lives:
blast_bin_dir = "# SET YOUR BLAST BIN PATH"

# 2) Prepend it to PATH (os.pathsep is ":" on *nix, ";" on Windows)
os.environ["PATH"] = blast_bin_dir + os.pathsep + os.environ.get("PATH", "")

In [ ]:
from pyblast.blastn.api import PyBlastnAPI

pybla = PyBlastnAPI()



In [ ]:
from Bio import SeqIO

recs = []

for record in SeqIO.parse('gencode.v48.transcripts.fa', 'fasta'):
    record.id = record.id.split('|')[4]
    record.description = record.id
    recs.append(record)

SeqIO.write(recs, 'gencode.v48.transcripts2.fa', 'fasta')

In [ ]:
%%bash
#wget https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_48/gencode.v48.transcripts.fa.gz
#gunzip gencode.v48.transcripts.fa.gz
rm blastdb/*

In [ ]:
fasta_path = 'gencode.v48.transcripts2.fa'
outfmt = "6 qseqid sseqid pident qcovhsp length qlen mismatch gapopen qstart qend sstart send evalue bitscore"
#db_path = 'blastdb/2aef5b1d-0d40-40da-b9e5-4e0d73cdf040' # made from fasta_path


df = pybla.blast_fasta_against_fasta(
    'Ahsa_siRNA.fasta', fasta_path, outfmt=outfmt,
    num_threads=4, word_size=10, max_hsps=30, perc_identity=70, qcov_hsp_perc=70,
)

df

In [ ]:
no_ahsa = df[df.sseqid.str.contains('AHSA1') == False]
no_ahsa = no_ahsa[no_ahsa.sseqid.str.contains('ENST') == False]
no_ahsa['sseqid'] = no_ahsa['sseqid'].map(lambda x: x.split('-')[0])
no_ahsa['hit_perc'] = (no_ahsa['pident'] * no_ahsa['qcovhsp'] / 100).round(2)
no_ahsa = no_ahsa[no_ahsa.hit_perc > 80].sort_values(by='hit_perc', ascending=False).drop_duplicates(subset=['qseqid', 'sseqid'])
no_ahsa

In [ ]:
gene_hits = no_ahsa.sseqid.value_counts()
sns.barplot(gene_hits[gene_hits > 2])
plt.xticks(rotation=45, ha='right')

In [ ]:
no_ahsa.qseqid.value_counts()

In [ ]:
keys = {k: i for i,k in enumerate(no_ahsa.columns)}
si_hit_map = {}
for row in no_ahsa.values:
    si = row[keys['qseqid']]
    gene_hit = row[keys['sseqid']]
    hit_perc = row[keys['hit_perc']]
    form = f"{gene_hit}-{hit_perc}%; "
    
    if si in si_hit_map:
        si_hit_map[si] += form
    else:
        si_hit_map[si] = form

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

final_design['cross_reactive_hits'] = final_design['siRNA'].map(si_hit_map)
final_design['cross_reactive_count'] = final_design['cross_reactive_hits'].astype(str).str.count(';')
sns.histplot(data= final_design, x='cross_reactive_count', bins=20)
plt.ylabel('siRNA #')

In [ ]:
final_design.to_csv(f'Ahsa1-{best_pair}_pattern_design_siRNA_N=85_with_cross_reactivity.csv')

In [ ]:
plt.figure(dpi=200)
sns.histplot(final_design.mean_pred, bins=50)
plt.xlabel('Mean Predicted Activity % - from 10 Seed Models Full Model with Thermo')
plt.ylabel('siRNA #')

In [ ]:
plt.figure(dpi=200)
sns.histplot(final_design.siRNA_percent_loc, bins=50)
plt.xlabel('mRNA location normalized 0-1')
plt.ylabel('siRNA #')

In [ ]:
final_design.columns